In [6]:
from pyspark import SparkContext

In [7]:
from pyspark.sql import SparkSession

In [8]:

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [9]:
sc = spark.sparkContext

In [10]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema() #schemat - w jaki sposob dane, tu data frame
#timestamp trzeba zmienic typ, bo to ma byc przetwarzanie w czasie rzeczywistym

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [11]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [12]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [13]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum

df.groupBy("category") \
  .agg(
      _sum("amount").alias("suma"),
      _min("amount").alias("minimum"),
      _max("amount").alias("maksimum")
  ) \
  .orderBy("category") \
  .show()

+-----------+------------------+-------+--------+
|   category|              suma|minimum|maksimum|
+-----------+------------------+-------+--------+
|elektronika|1520770.6899999995|    9.0|  9999.0|
|    książki| 851382.0799999991|    5.0| 9107.25|
|     odzież| 849877.5500000007|    5.0| 9696.63|
|    żywność| 789514.4300000003|    5.0| 6916.92|
+-----------+------------------+-------+--------+



In [ ]:
######### grupowanie po przedziale czasowym w oknach

In [17]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [18]:
hourly.printSchema # pierwsza kolumna - w python nie moze byc kolumny ktora ma w srodku jakies elemnty, tu moze bo spark

<bound method DataFrame.printSchema of DataFrame[window: struct<start:timestamp,end:timestamp>, liczba_tx: bigint, suma_PLN: double]>

In [17]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
) 
### wyciagam z tej jednej kolumny i robie dwie 

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [20]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [21]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: sliding ma więcej wierszy, ponieważ ma więcej okien czasowych; jedna transakcja może należeć do kilku okien

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [24]:
# praca domowa  - 1 polecenie
from pyspark.sql.functions import window, avg, col

result = (
    df.filter(col("store") == "Gdańsk")
      .groupBy(window("timestamp", "1 hour"))
      .agg(avg("amount").alias("srednia_kwota"))
      .orderBy("srednia_kwota"))  

result.show(1, truncate=False)

+------------------------------------------+-----------------+
|window                                    |srednia_kwota    |
+------------------------------------------+-----------------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.0118407310706|
+------------------------------------------+-----------------+
only showing top 1 row



In [32]:
from pyspark.sql.functions import window, count, col

tumbling_1 = (
    df.groupBy(
        window("timestamp", "30 minutes"),
        col("category")  
    )
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "category",
        "liczba_tx"
    )
    .orderBy("od")
)

tumbling_1.show()

+-------------------+-------------------+-----------+---------+
|                 od|                 do|   category|liczba_tx|
+-------------------+-------------------+-----------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|elektronika|      269|
|2026-04-12 08:00:00|2026-04-12 08:30:00|     odzież|      302|
|2026-04-12 08:00:00|2026-04-12 08:30:00|    książki|      268|
|2026-04-12 08:00:00|2026-04-12 08:30:00|    żywność|      273|
|2026-04-12 08:30:00|2026-04-12 09:00:00|    książki|      527|
|2026-04-12 08:30:00|2026-04-12 09:00:00|     odzież|      506|
|2026-04-12 08:30:00|2026-04-12 09:00:00|    żywność|      486|
|2026-04-12 08:30:00|2026-04-12 09:00:00|elektronika|      519|
|2026-04-12 09:00:00|2026-04-12 09:30:00|elektronika|      611|
|2026-04-12 09:00:00|2026-04-12 09:30:00|     odzież|      605|
|2026-04-12 09:00:00|2026-04-12 09:30:00|    książki|      622|
|2026-04-12 09:00:00|2026-04-12 09:30:00|    żywność|      567|
|2026-04-12 09:30:00|2026-04-12 10:00:00

In [33]:
from pyspark.sql.functions import col

result = (
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") <  "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .count()
    .orderBy("category")
)

result.show()

+-----------+-----+
|   category|count|
+-----------+-----+
|elektronika|  611|
|    książki|  622|
|     odzież|  605|
|    żywność|  567|
+-----------+-----+



In [34]:
from pyspark.sql.functions import window, count

result = (
    df.groupBy(window("timestamp", "15 minutes"))
      .agg(count("tx_id").alias("liczba_tx"))
      .orderBy("liczba_tx", ascending=False)
)

result.show(1, truncate=False)

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
+------------------------------------------+---------+
only showing top 1 row

